# Pré-processamento, Comparativo e Otimização

Este notebook representa a etapa  de processamento de linguagem natural (NLP).

Vamos configurar 4 pipelines independentes simultaneamente que atuarão sobre nosso `dataset.csv` com o intuito de comparar as abordagens:

- **Simples (Limpeza Regex Básica):** Remove apenas pontuações, provando se complexidade computacional é realmente necessária.
- **NLTK (Stemming):** Focado historicamente na pesquisa. Elimina o sufixo das palavras. Altamente leve, mas desfigura a língua.
- **spaCy (Lemmatization):** Focado não em quebrar, mas em entender a classe gramatical (`Part-of-Speech`) da palavra no contexto e transformá-la no seu estado de dicionário (`fui` para `ir`). Retém 100% de significado.
- **Embeddings Densos:** Extrai do modelo interno do spaCy os vetores flutuantes latentes.Essas arrays capturam similaridades abstratas, sofrendo forte custo de performance.

## 1. Importação das Dependências

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import time

nltk.download('stopwords', quiet=True)
stemmer = SnowballStemmer('portuguese')
stop_words = set(stopwords.words('portuguese'))

nlp = spacy.load("pt_core_news_lg")

In [2]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

## 2. Carregamento dos Dados Brutos


In [3]:
df = pd.read_csv('data/dataset.csv')

In [4]:
df.head(2)

,id_registro,texto,canal_origem,data,classe_macro,classe_detalhada
0,1,Erro recorrente no sistema ao tentar acessar d...,sistema,2025-03-08,Problemas Técnicos,Erro de Sistema / Aplicação
1,2,Pedido de integração para sincronização de dados.,email,2025-03-22,Solicitações Operacionais,Integração com Sistemas Externos


## 3. Aplicação do Framework de Processamentos
Criando os competidores.

In [5]:
def pipeline_simples(texto):
    texto = str(texto).lower().strip()
    return re.sub(r'[^\w\s]', '', texto)

def pipeline_nltk(texto):
    texto = pipeline_simples(texto)
    tokens = texto.split()
    tokens_stemmed = [stemmer.stem(t) for t in tokens if t not in stop_words]
    return " ".join(tokens_stemmed)

def pipeline_spacy(texto):
    doc = nlp(str(texto).lower().strip())
    tokens_lemmatized = [token.lemma_ for token in doc if not token.is_punct and not token.is_stop and not token.is_space]
    return " ".join(tokens_lemmatized)

df['texto_simples'] = df['texto'].apply(pipeline_simples)
df['texto_nltk'] = df['texto'].apply(pipeline_nltk)
df['texto_spacy'] = df['texto'].apply(pipeline_spacy)
df['embedding'] = df['texto'].apply(lambda x: nlp(str(x)).vector)

display(df[['texto', 'texto_simples', 'texto_nltk', 'texto_spacy', 'embedding']].tail(3))

,texto,texto_simples,texto_nltk,texto_spacy,embedding
1197,O login está falhando e impede o acesso ao amb...,o login está falhando e impede o acesso ao amb...,login falh imped acess ambient,login falhar impedir acesso ambiente,"[1.7887344, -0.80996186, -0.074311905, 2.24402..."
1198,Solicito a criação de um novo usuário no sistema.,solicito a criação de um novo usuário no sistema,solicit criaçã nov usuári sistem,solicitar criação usuário,"[0.740492, -0.16826399, -0.534918, 2.0238028, ..."
1199,Mensagem sem detalhes específicos.,mensagem sem detalhes específicos,mensag detalh específ,mensagem detalhe específico,"[0.487284, -1.405012, 0.533738, 0.905764, -3.3..."


## 4. O Impacto Físico do Processamento:

In [6]:
# 1) Verificando redução
vocab_original = set(" ".join(df['texto'].str.lower()).split())
vocab_simples = set(" ".join(df['texto_simples']).split())
vocab_nltk = set(" ".join(df['texto_nltk']).split())
vocab_spacy = set(" ".join(df['texto_spacy']).split())

In [7]:

print("ANÁLISE DE DIMENSIONALIDADE (ESPAÇO DO DICIONÁRIO)")
print("\n")

print(f"Vocabulário Original Bruto         : {len(vocab_original)} termos únicos")
print(f"Vocabulário Regex (Limpo Simples)  : {len(vocab_simples)} termos únicos")
print(f"Vocabulário NLTK (Stemming)        : {len(vocab_nltk)} termos únicos")
print(f"Vocabulário spaCy (Lemmatization)  : {len(vocab_spacy)} termos únicos")

ANÁLISE DE DIMENSIONALIDADE (ESPAÇO DO DICIONÁRIO)


Vocabulário Original Bruto         : 260 termos únicos
Vocabulário Regex (Limpo Simples)  : 241 termos únicos
Vocabulário NLTK (Stemming)        : 171 termos únicos
Vocabulário spaCy (Lemmatization)  : 176 termos únicos


In [8]:
# 2) Teste Crítico de Velocidade em Millisegundos
start = time.time(); df['texto'].apply(pipeline_simples); t_simples = time.time() - start
start = time.time(); df['texto'].apply(pipeline_nltk); t_nltk = time.time() - start
start = time.time(); df['texto'].apply(pipeline_spacy); t_spacy = time.time() - start
start = time.time(); df['texto'].apply(lambda x: nlp(str(x)).vector); t_emb = time.time() - start

In [9]:
print("TESTE DE VELOCIDADE ")
print("\n")

print(f"Execução de Pipeline Simples       : {t_simples:.4f} s")
print(f"Execução de Pipeline NLTK          : {t_nltk:.4f} s")
print(f"Execução de Pipeline spaCy (Lemma) : {t_spacy:.4f} s")
print(f"Geração de spaCy Embeddings: {t_emb:.4f} s")

TESTE DE VELOCIDADE 


Execução de Pipeline Simples       : 0.0022 s
Execução de Pipeline NLTK          : 0.0742 s
Execução de Pipeline spaCy (Lemma) : 4.2201 s
Geração de spaCy Embeddings: 4.2031 s


> `spaCy` se provou muito eficiente porque condensou formas verbais em raízes saudáveis, reduzindo o número de palavras pro classificador mantendo a legibilidade linguística.

> Forçar um Embeddings cego eleva absurdamente o tempo de processamento comparado à expressões regulares rasas.

## 5. Salvar Dataframe e Otimização para Modelagem


> A partir dos testes cruzados, a lematização avançada do **SpaCy** apresenta a melhor correlação de representatividade semântica vs. redução de ortografia. A etapa puramente reativa `texto_simples` (Apenas RegEx) não atinge o mesmo nível relacional de generalização de tempos verbais, não justificando o fardo na memória. 

> Por essa razão, para a modelagem corporativa eliminaremos as colunas intermediárias (simples e nltk), preservando exclusivamente a varíavel refinada e o embedding.

In [13]:
colunas_essenciais = ['id_registro', 'texto', 'texto_spacy', 'embedding', 'canal_origem', 'data', 'classe_macro', 'classe_detalhada']
df_final = df[colunas_essenciais].copy()

df_final.to_pickle('data/dataset_processed.pkl')
print("Dataset Salvo! ('data/dataset_processed.pkl')")

Dataset Salvo! ('data/dataset_processed.pkl')


In [14]:
df_final.head(2)

,id_registro,texto,texto_spacy,embedding,canal_origem,data,classe_macro,classe_detalhada
0,1,Erro recorrente no sistema ao tentar acessar d...,erro recorrente acessar determinar funcionalidade,"[0.8984845, -0.67061794, 0.3228, 1.778956, -0....",sistema,2025-03-08,Problemas Técnicos,Erro de Sistema / Aplicação
1,2,Pedido de integração para sincronização de dados.,pedir integração sincronização dado,"[1.3615377, 0.4630312, -0.63453364, 0.3071998,...",email,2025-03-22,Solicitações Operacionais,Integração com Sistemas Externos


> Para a modelagem final, optamos por manter tanto o processamento **TF-IDF/spaCy** quanto os **Embeddings**. Dado que o dataset possui textos curtos e alta redundância, essa abordagem dupla é fundamental para isolar o pipeline de vieses e garantir a generalização do classificador. O impacto no tempo de processamento é aceitável diante do ganho em robustez estatística que a diversidade de features trará ao modelo.